# Pipeline step 1: stratified culture-vs-language diagnosis

Goal: measure the **culture** effect on jailbreak success (`us` / `japan` vs `neutral`)
**controlling for language**, instead of pooling `en`/`ja`/`mix` together. Language
becomes the stratifying ("control") variable; culture is the exposure of interest.

**Method: Cochran-Mantel-Haenszel (CMH).** For each language stratum, build a 2x2 table
(culture exposed vs not) x (unsafe vs not), then pool across strata into one combined
odds ratio. This is the standard epidemiology tool for exactly this situation — "does X
affect Y, after accounting for a third variable Z that might confound it" — pulled in
here instead of the plain pooled logistic regression from `stat_analysis.ipynb`, which
mixed language into the same model as culture rather than using it as a control.

If the per-language (`en`, `ja`) odds ratios roughly agree *and* the pooled MH estimate
is significant, that's good evidence of a real, language-independent cultural effect. If
they disagree a lot, the "culture" signal is partly a language artifact.

No GPU needed — this is pandas/scipy on the already-loaded `observations.csv`.


## 0. Load data (same Drive-detection as the other notebooks)

In [1]:

import glob
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "chat.py").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

LOCAL_DIR = None
DRIVE_RELATIVE = Path("ez-penta/dataset2_eval_gguf/combined")
DRIVE_CANDIDATES = [
    Path("/content/drive/MyDrive") / DRIVE_RELATIVE,
    *[Path(p) / "My Drive" / DRIVE_RELATIVE
      for p in glob.glob(str(Path.home() / "Library/CloudStorage/GoogleDrive-*"))],
    Path.home() / "Google Drive" / "My Drive" / DRIVE_RELATIVE,
]
if LOCAL_DIR:
    COMBINED_DIR = (Path.cwd() / LOCAL_DIR) if not Path(LOCAL_DIR).is_absolute() else Path(LOCAL_DIR)
else:
    COMBINED_DIR = next((p for p in DRIVE_CANDIDATES if p.exists()), None)
    if COMBINED_DIR is None and Path("/content").exists():
        from google.colab import drive
        drive.mount("/content/drive")
        COMBINED_DIR = Path("/content/drive/MyDrive") / DRIVE_RELATIVE
    if COMBINED_DIR is None:
        raise FileNotFoundError(
            "Could not find the Drive-synced `combined` folder. Install/sign in to "
            "Google Drive for Desktop, run this notebook in Colab, or set LOCAL_DIR."
        )
print(f"Reading combined results from: {COMBINED_DIR}")

observations = pd.read_csv(COMBINED_DIR / "observations.csv", low_memory=False)
usable = observations[
    (observations["analysis_eligible"] == True)
    & (observations["generation_status"] == "completed")
    & (observations["scoring_status"] == "completed")
    & (observations["split"] != "heldout")
].copy()
harmful = usable[usable["kind"] == "harmful"].copy()
harmful["unsafe"] = harmful["safety"].eq("Unsafe").astype(int)
print(f"{len(harmful)} usable harmful-seed rows")


Reading combined results from: /Users/isaac/Library/CloudStorage/GoogleDrive-test-ezpenta/My Drive/ez-penta/dataset2_eval_gguf/combined


4069 usable harmful-seed rows


## 1. Mantel-Haenszel pooling function

Standard formulas (Mantel & Haenszel 1959; variance per Robins, Breslow & Greenland
1986): pooled odds ratio, MH chi-square test (with continuity correction), and a 95%
CI on the log odds ratio.

In [2]:

def two_by_two(df, exposure_mask, outcome_col):
    a = int((exposure_mask & (df[outcome_col] == 1)).sum())
    b = int((exposure_mask & (df[outcome_col] == 0)).sum())
    c = int((~exposure_mask & (df[outcome_col] == 1)).sum())
    d = int((~exposure_mask & (df[outcome_col] == 0)).sum())
    return a, b, c, d


def mantel_haenszel(df, exposure_col, exposure_value, baseline_value, outcome_col, stratum_col):
    # Pool the (exposure_value vs baseline_value) odds ratio for `outcome_col` across
    # strata of `stratum_col`. Returns a dict with per-stratum and pooled results.
    strata = sorted(df[stratum_col].dropna().unique())
    per_stratum = []
    num_or, den_or = 0.0, 0.0  # for OR_MH
    sum_a, sum_Ea, sum_Var = 0.0, 0.0, 0.0  # for the MH chi-square
    sum_R, sum_S, sum_P_R, sum_P_S, sum_Q_R, sum_Q_S = (0.0,) * 6  # for RBG variance

    for stratum in strata:
        sub = df[(df[stratum_col] == stratum) & (df[exposure_col].isin([exposure_value, baseline_value]))]
        exposure_mask = sub[exposure_col] == exposure_value
        a, b, c, d = two_by_two(sub, exposure_mask, outcome_col)
        n = a + b + c + d
        if n == 0 or (a + c) == 0 or (b + d) == 0:
            continue
        or_stratum = (a * d) / (b * c) if b * c > 0 else np.inf
        p_stratum = stats.fisher_exact([[a, b], [c, d]])[1]
        per_stratum.append({"stratum": stratum, "a_exposed_unsafe": a, "b_exposed_safe": b,
                              "c_unexposed_unsafe": c, "d_unexposed_safe": d, "n": n,
                              "odds_ratio": or_stratum, "fisher_p": p_stratum})

        num_or += (a * d) / n
        den_or += (b * c) / n
        Ea = (a + b) * (a + c) / n
        Va = (a + b) * (c + d) * (a + c) * (b + d) / (n**2 * (n - 1)) if n > 1 else 0
        sum_a += a
        sum_Ea += Ea
        sum_Var += Va

        R, S = (a * d) / n, (b * c) / n
        P, Q = (a + d) / n, (b + c) / n
        sum_R += R
        sum_S += S
        sum_P_R += P * R
        sum_P_S += P * S
        sum_Q_R += Q * R
        sum_Q_S += Q * S

    or_mh = num_or / den_or if den_or > 0 else np.nan
    chi2_mh = (abs(sum_a - sum_Ea) - 0.5) ** 2 / sum_Var if sum_Var > 0 else np.nan
    p_mh = stats.chi2.sf(chi2_mh, df=1) if np.isfinite(chi2_mh) else np.nan

    # Robins-Breslow-Greenland variance of ln(OR_MH)
    if sum_R > 0 and sum_S > 0:
        var_ln_or = (sum_P_R / (2 * sum_R**2)) + ((sum_P_S + sum_Q_R) / (2 * sum_R * sum_S)) + (sum_Q_S / (2 * sum_S**2))
        se_ln_or = np.sqrt(var_ln_or)
        ci_lo = np.exp(np.log(or_mh) - 1.96 * se_ln_or)
        ci_hi = np.exp(np.log(or_mh) + 1.96 * se_ln_or)
    else:
        ci_lo = ci_hi = np.nan

    return {
        "exposure": f"{exposure_col}={exposure_value} vs {baseline_value}",
        "or_mh": or_mh, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "chi2_mh": chi2_mh, "p_mh": p_mh,
        "per_stratum": pd.DataFrame(per_stratum),
    }


## 2. Culture effect, pooled across `en`/`ja`/`mix` (language as the control variable)

Two exposures tested: `culture=us` vs `neutral`, and `culture=japan` vs `neutral`.

In [3]:

results = {}
for culture_value in ["us", "japan"]:
    result = mantel_haenszel(harmful, "culture", culture_value, "neutral", "unsafe", "prompt_language")
    results[culture_value] = result
    print(f"=== {result['exposure']}, pooled across prompt_language strata ===")
    print(f"OR_MH = {result['or_mh']:.3f}  (95% CI {result['ci_lo']:.3f}-{result['ci_hi']:.3f})"
          f"  chi2_MH={result['chi2_mh']:.2f}  p={result['p_mh']:.6f}")
    display(result["per_stratum"].round(4))
    print()


=== culture=us vs neutral, pooled across prompt_language strata ===
OR_MH = 1.982  (95% CI 1.583-2.482)  chi2_MH=35.78  p=0.000000


,stratum,a_exposed_unsafe,b_exposed_safe,c_unexposed_unsafe,d_unexposed_safe,n,odds_ratio,fisher_p
0,en,55,350,30,356,791,1.8648,0.0112
1,ja,90,325,47,322,784,1.8972,0.0010
2,mix,117,428,57,442,1044,2.1198,0.0000



=== culture=japan vs neutral, pooled across prompt_language strata ===
OR_MH = 1.097  (95% CI 0.862-1.397)  chi2_MH=0.48  p=0.487465


,stratum,a_exposed_unsafe,b_exposed_safe,c_unexposed_unsafe,d_unexposed_safe,n,odds_ratio,fisher_p
0,en,44,407,30,356,837,1.2829,0.3308
1,ja,54,363,47,322,786,1.0192,1.0000
2,mix,70,512,57,442,1081,1.0602,0.7770


## 3. Does the culture effect replicate in `en` AND `ja` separately?

This is the actual "control for language" check: compare the per-stratum odds ratios
(from the table above) between `en` and `ja`. Consistent sign and similar magnitude in
both = a real cultural effect. The `mix` stratum is shown too, as a third (weaker,
code-switched) data point, but `en`/`ja` are the two that matter for this check.

In [4]:

for culture_value, result in results.items():
    per_stratum = result["per_stratum"].set_index("stratum")
    print(f"--- culture={culture_value} vs neutral: per-language odds ratio and Fisher p ---")
    for lang in ["en", "ja", "mix"]:
        if lang in per_stratum.index:
            row = per_stratum.loc[lang]
            print(f"  {lang}: OR={row['odds_ratio']:.3f}  p={row['fisher_p']:.5f}  n={int(row['n'])}")
    en_or = per_stratum.loc["en", "odds_ratio"] if "en" in per_stratum.index else np.nan
    ja_or = per_stratum.loc["ja", "odds_ratio"] if "ja" in per_stratum.index else np.nan
    same_direction = (en_or > 1) == (ja_or > 1)
    print(f"  en and ja agree on direction (both > 1 or both < 1): {same_direction}")
    print()


--- culture=us vs neutral: per-language odds ratio and Fisher p ---
  en: OR=1.865  p=0.01117  n=791
  ja: OR=1.897  p=0.00097  n=784
  mix: OR=2.120  p=0.00001  n=1044
  en and ja agree on direction (both > 1 or both < 1): True

--- culture=japan vs neutral: per-language odds ratio and Fisher p ---
  en: OR=1.283  p=0.33084  n=837
  ja: OR=1.019  p=1.00000  n=786
  mix: OR=1.060  p=0.77699  n=1081
  en and ja agree on direction (both > 1 or both < 1): True



## 4. Same check, restricted to `framing=rp` (the known worst framing)

The earlier interaction test found `culture x framing` significant overall. This
repeats the stratified-by-language check specifically within the `rp` (roleplay)
framing, to see whether the `us+rp` combination is consistent across languages too.

In [5]:

rp_only = harmful[harmful["framing"] == "rp"]
print(f"{len(rp_only)} rows with framing=rp")

rp_results = {}
for culture_value in ["us", "japan"]:
    result = mantel_haenszel(rp_only, "culture", culture_value, "neutral", "unsafe", "prompt_language")
    rp_results[culture_value] = result
    print(f"\n=== framing=rp only: {result['exposure']}, pooled across language ===")
    print(f"OR_MH = {result['or_mh']:.3f}  (95% CI {result['ci_lo']:.3f}-{result['ci_hi']:.3f})"
          f"  chi2_MH={result['chi2_mh']:.2f}  p={result['p_mh']:.6f}")
    display(result["per_stratum"].round(4))


1075 rows with framing=rp

=== framing=rp only: culture=us vs neutral, pooled across language ===
OR_MH = 2.544  (95% CI 1.735-3.731)  chi2_MH=22.90  p=0.000002


,stratum,a_exposed_unsafe,b_exposed_safe,c_unexposed_unsafe,d_unexposed_safe,n,odds_ratio,fisher_p
0,en,24,62,8,75,169,3.6290,0.0030
1,ja,26,73,15,67,181,1.5909,0.2171
2,mix,58,129,23,146,356,2.8541,0.0001



=== framing=rp only: culture=japan vs neutral, pooled across language ===
OR_MH = 1.422  (95% CI 0.945-2.138)  chi2_MH=2.52  p=0.112094


,stratum,a_exposed_unsafe,b_exposed_safe,c_unexposed_unsafe,d_unexposed_safe,n,odds_ratio,fisher_p
0,en,17,79,8,75,179,2.0174,0.1352
1,ja,15,69,15,67,166,0.9710,1.0000
2,mix,36,153,23,146,358,1.4936,0.1993


## 5. Save the pooled estimates for Notebook 5 (before/after validation)

Notebook 5 will recompute this exact statistic on the post-fix model's outputs and
compare against these baseline numbers.

In [6]:

import json

baseline_summary = {
    culture_value: {"or_mh": result["or_mh"], "ci_lo": result["ci_lo"], "ci_hi": result["ci_hi"],
                     "p_mh": result["p_mh"]}
    for culture_value, result in results.items()
}
baseline_summary_rp = {
    culture_value: {"or_mh": result["or_mh"], "ci_lo": result["ci_lo"], "ci_hi": result["ci_hi"],
                     "p_mh": result["p_mh"]}
    for culture_value, result in rp_results.items()
}
out = {"overall": baseline_summary, "framing_rp_only": baseline_summary_rp}
out_path = COMBINED_DIR / "pipeline1_baseline_mh_stats.json"
out_path.write_text(json.dumps(out, indent=2, default=float))
print(f"Saved baseline MH statistics to {out_path}")
print(json.dumps(out, indent=2, default=float))


Saved baseline MH statistics to /Users/isaac/Library/CloudStorage/GoogleDrive-test-ezpenta/My Drive/ez-penta/dataset2_eval_gguf/combined/pipeline1_baseline_mh_stats.json
{
  "overall": {
    "us": {
      "or_mh": 1.9822024722961555,
      "ci_lo": 1.5831753321715596,
      "ci_hi": 2.481801327581078,
      "p_mh": 2.2075405470463405e-09
    },
    "japan": {
      "or_mh": 1.0974400938756737,
      "ci_lo": 0.8623254476955374,
      "ci_hi": 1.3966591880877415,
      "p_mh": 0.48746508677598244
    }
  },
  "framing_rp_only": {
    "us": {
      "or_mh": 2.5441397524257483,
      "ci_lo": 1.734877292550024,
      "ci_hi": 3.7308961894123778,
      "p_mh": 1.7042105405251645e-06
    },
    "japan": {
      "or_mh": 1.421704273392712,
      "ci_lo": 0.945202051537464,
      "ci_hi": 2.138424305888195,
      "p_mh": 0.11209432929610925
    }
  }
}
